<a href="https://colab.research.google.com/github/JuliaMcPhillips/ds2002-fa26/blob/main/Copy_of_2026_09_23_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
print('shape:', df.shape)

print('\ndtypes:')
print(df.dtypes)

print('\nnull counts:')
print(df.isna().sum())

print('\nexact duplicate rows:', df.duplicated().sum())

shape: (8, 6)

dtypes:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

null counts:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [6]:
removed = df.duplicated().sum()

clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [7]:
# TODO: clean['price'] = ...

# assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)

assert clean['price'].dtype == float

log(
    'price',
    'stripped dollar signs/whitespace and converted price from text to float',
    len(clean)
)

[price] stripped dollar signs/whitespace and converted price from text to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [8]:
# TODO: clean['qty'] = pd.to_numeric(...)

clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

# Decision 1: drop rows with missing quantity because units/revenue cannot be determined reliably.
clean = clean.loc[clean['qty'].notna()].copy()

log(
    'quantity-missing',
    'dropped rows with missing quantity because quantity cannot be inferred',
    missing
)

# Decision 2: keep negative quantities because they represent refunds/returns
# and should reduce net revenue.
log(
    'quantity-negative',
    'kept negative quantities as refunds/returns',
    negative
)

# TODO: apply your decision, then log both separately

[quantity-missing] dropped rows with missing quantity because quantity cannot be inferred (1 row(s))
[quantity-negative] kept negative quantities as refunds/returns (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [9]:
print('before:', sorted(clean['category'].unique()))

before_n = clean['category'].nunique()

clean['category'] = (
    clean['category']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '', regex=True)
)

CATEGORY_MAP = {
    'food': 'food',
    'merch': 'merch',
    'apparel': 'apparel',
    'raingear': 'rain gear'
}

clean['category'] = clean['category'].map(CATEGORY_MAP).fillna(clean['category'])

after_n = clean['category'].nunique()

print('after: ', sorted(clean['category'].unique()))

log(
    'categories',
    f'normalized category labels from {before_n} distinct values to {after_n}',
    before_n - after_n
)

# TODO: lowercase, strip, remove punctuation
# TODO: CATEGORY_MAP = {...} for the judgment calls

# print('after: ', sorted(clean['category'].unique()))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'rain gear']
[categories] normalized category labels from 6 distinct values to 4 (2 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [10]:
before_items = clean['item'].nunique(dropna=True)

clean['item'] = (
    clean['item']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '', regex=True)
)

ITEM_MAP = {
    'cheeseburger': 'cheeseburger',
    'foamfinger': 'foam finger',
    'uvatshirt': 'uva t-shirt',
    'rainponcho': 'rain poncho'
}

clean['item'] = clean['item'].map(ITEM_MAP).fillna(clean['item'])

missing_item = clean['item'].isna().sum()

clean = clean.loc[clean['item'].notna()].copy()

after_items = clean['item'].nunique(dropna=True)

log(
    'items',
    f'normalized item names from {before_items} distinct non-null values to {after_items}',
    before_items - after_items
)

log(
    'item-missing',
    'dropped rows with missing item name because the product cannot be identified',
    missing_item
)

print('items:', sorted(clean['item'].unique()))

[items] normalized item names from 5 distinct non-null values to 3 (2 row(s))
[item-missing] dropped rows with missing item name because the product cannot be identified (1 row(s))
items: ['cheeseburger', 'rain poncho', 'uva t-shirt']


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [11]:
clean['ts'] = pd.to_datetime(
    clean['ts'],
    errors='coerce',
    format='mixed'
)

failed_ts = clean['ts'].isna().sum()

print('timestamp parse failures/missing:', failed_ts)

clean['hour'] = clean['ts'].dt.hour

log(
    'timestamps',
    'parsed timestamps; kept missing/unparseable timestamps as NaT',
    failed_ts
)

timestamp parse failures/missing: 1
[timestamps] parsed timestamps; kept missing/unparseable timestamps as NaT (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [12]:
assert clean.duplicated().sum() == 0
assert pd.api.types.is_float_dtype(clean['price'])
assert pd.api.types.is_numeric_dtype(clean['qty'])
assert clean['qty'].notna().all()
assert clean['item'].notna().all()
assert set(clean['category']) == {'food', 'apparel', 'rain gear'}
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])

clean['revenue'] = clean['qty'] * clean['price']

print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 6.0
revenue: 76.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [13]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,NaN
1,duplicates,dropped exact duplicate rows,1.0
2,price,stripped dollar signs/whitespace and converted...,7.0
3,quantity-missing,dropped rows with missing quantity because qua...,1.0
4,quantity-negative,kept negative quantities as refunds/returns,1.0
5,categories,normalized category labels from 6 distinct val...,2.0
6,items,normalized item names from 5 distinct non-null...,2.0
7,item-missing,dropped rows with missing item name because th...,1.0
8,timestamps,parsed timestamps; kept missing/unparseable ti...,1.0


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [14]:
# Checkpoint
rows_after = len(clean)
revenue_after = clean['revenue'].sum()

biggest_decision = 'kept the negative quantity as a refund/return'

revenue_other_way = clean.loc[
    clean['qty'] >= 0,
    'revenue'
].sum()

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: kept the negative quantity as a refund/return
revenue the other way: 94.5
